# Reconhecimento de Gestos Personalizados 🎯
Este notebook utiliza o modelo **Random Forest** que você treinou (`gesture_classifier.pkl`) para reconhecer seus próprios gestos em tempo real via webcam.

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pickle
import os

# 1. Caminhos dos arquivos
MODEL_PKL_PATH = "../storage/nn_models/gesture_classifier.pkl"
MEDIAPIPE_TASK_PATH = '../storage/nn_models/gesture_recognizer.task'

# 2. Carregar o modelo treinado (Scikit-Learn)
if not os.path.exists(MODEL_PKL_PATH):
    raise FileNotFoundError(f"Modelo personalizado não encontrado em {MODEL_PKL_PATH}. Treine o modelo primeiro!")

with open(MODEL_PKL_PATH, 'rb') as f:
    custom_model = pickle.load(f)
print("Modelo personalizado carregado com sucesso!")

# 3. Configurar o MediaPipe (apenas para extrair os landmarks)
base_options = python.BaseOptions(model_asset_path=MEDIAPIPE_TASK_PATH)
options = vision.GestureRecognizerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.7
)
recognizer = vision.GestureRecognizer.create_from_options(options)

## Funções de Processamento e Visualização

In [ ]:
def extract_features(hand_landmarks):
    """Converte os landmarks do MediaPipe para o formato que o modelo Random Forest espera."""
    features = []
    for lm in hand_landmarks:
        features.extend([lm.x, lm.y, lm.z])
    return np.array(features).reshape(1, -1)

def draw_manual_landmarks(image, hand_landmarks, label_text):
    h, w, _ = image.shape
    
    CONNECTIONS = [
        (0,1), (1,2), (2,3), (3,4), (0,5), (5,6), (6,7), (7,8),
        (5,9), (9,10), (10,11), (11,12), (9,13), (13,14), (14,15),
        (15,16), (13,17), (17,18), (18,19), (19,20), (0,17)
    ]

    # Desenhar conexões
    for start_idx, end_idx in CONNECTIONS:
        pt1 = (int(hand_landmarks[start_idx].x * w), int(hand_landmarks[start_idx].y * h))
        pt2 = (int(hand_landmarks[end_idx].x * w), int(hand_landmarks[end_idx].y * h))
        cv2.line(image, pt1, pt2, (255, 255, 255), 1)

    # Desenhar pontos
    for landmark in hand_landmarks:
        cx, cy = int(landmark.x * w), int(landmark.y * h)
        cv2.circle(image, (cx, cy), 3, (0, 255, 0), -1)

    # Mostrar o Gesto Identificado pelo SEU modelo
    tx, ty = int(hand_landmarks[0].x * w), int(hand_landmarks[0].y * h) - 20
    cv2.putText(image, f"Custom: {label_text}", (tx, ty), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2, cv2.LINE_AA)
    
    return image

## Execução em Tempo Real
Pressione **'q'** para sair.

In [ ]:
cap = cv2.VideoCapture(0)

print("Reconhecimento de gestos CUSTOMIZADOS iniciado...")

while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    
    timestamp = int(cv2.getTickCount() / cv2.getTickFrequency() * 1000)
    result = recognizer.recognize_for_video(mp_image, timestamp)

    if result.hand_landmarks:
        landmarks = result.hand_landmarks[0]
        
        # 1. Extrair os pontos para o formato do scikit-learn
        input_data = extract_features(landmarks)
        
        # 2. Fazer a predição usando o SEU modelo .pkl
        prediction = custom_model.predict(input_data)[0]
        probabilities = custom_model.predict_proba(input_data)
        confidence = np.max(probabilities)

        # 3. Visualizar
        label = f"{prediction} ({round(confidence*100, 1)}%)"
        frame = draw_manual_landmarks(frame, landmarks, label)

    cv2.imshow('Custom Gesture Recognition', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Finalizado.")